In [ ]:
# ============================================================
# NUTRIRATE AML - FEATURE ENGINEERING
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path(r"D:\NutriRate-AML")

DATA_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "nutrirate_ml_clean.parquet"
)

df = pd.read_parquet(DATA_FILE)

print(f"Dataset shape: {df.shape}")
display(df.head())

In [ ]:
# ============================================================
# FEATURES AND TARGET
# ============================================================

BASE_FEATURES = [
    "energy",
    "fat",
    "saturated_fat",
    "carbohydrates",
    "sugars",
    "protein",
    "fiber",
    "salt"
]

TARGET = "grade"

X_base = df[BASE_FEATURES].copy()
y = df[TARGET].copy()

print("Base features:")
for feature in BASE_FEATURES:
    print(" -", feature)

print("\nTarget:", TARGET)

print("\nClass distribution:")
display(y.value_counts().sort_index())

In [ ]:
# ============================================================
# ADVANCED NUTRITIONAL FEATURE ENGINEERING
# ============================================================

X = X_base.copy()

EPSILON = 1e-6

# ------------------------------------------------------------
# Ratio-based features
# ------------------------------------------------------------

X["saturated_fat_ratio"] = (
    X["saturated_fat"] /
    (X["fat"] + EPSILON)
)

X["sugar_carb_ratio"] = (
    X["sugars"] /
    (X["carbohydrates"] + EPSILON)
)

X["fiber_carb_ratio"] = (
    X["fiber"] /
    (X["carbohydrates"] + EPSILON)
)

X["protein_carb_ratio"] = (
    X["protein"] /
    (X["carbohydrates"] + EPSILON)
)

# ------------------------------------------------------------
# Energy-normalized features
# ------------------------------------------------------------

X["protein_energy_ratio"] = (
    X["protein"] /
    (X["energy"] + EPSILON)
)

X["fat_energy_ratio"] = (
    X["fat"] /
    (X["energy"] + EPSILON)
)

X["sugar_energy_ratio"] = (
    X["sugars"] /
    (X["energy"] + EPSILON)
)

X["fiber_energy_ratio"] = (
    X["fiber"] /
    (X["energy"] + EPSILON)
)

X["salt_energy_ratio"] = (
    X["salt"] /
    (X["energy"] + EPSILON)
)

print("Original features :", len(BASE_FEATURES))
print("Engineered features:", len(X.columns) - len(BASE_FEATURES))
print("Total features     :", len(X.columns))

display(X.head())

In [ ]:
# ============================================================
# ENGINEERED FEATURE QUALITY CHECK
# ============================================================

engineered_features = [
    col for col in X.columns
    if col not in BASE_FEATURES
]

feature_check = X[engineered_features].describe().T

feature_check["missing"] = X[engineered_features].isna().sum()
feature_check["infinite"] = np.isinf(
    X[engineered_features]
).sum()

display(feature_check)

In [ ]:
# ============================================================
# HANDLE INFINITE VALUES
# ============================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

print(
    "Total missing values after ratio creation:",
    X.isna().sum().sum()
)

display(
    X.isna().sum()
    .sort_values(ascending=False)
    .head(20)
)

In [ ]:
# ============================================================
# MISSING VALUE CHECK
# ============================================================

missing_features = X.isna().sum()

print("Features containing missing values:")
display(
    missing_features[
        missing_features > 0
    ].sort_values(ascending=False)
)

In [ ]:
# ============================================================
# STRATIFIED DATA SPLIT
# ============================================================

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("DATASET SPLIT")
print("=" * 60)

print(f"Training   : {len(X_train):,}")
print(f"Validation : {len(X_val):,}")
print(f"Testing    : {len(X_test):,}")
print(
    f"Total      : "
    f"{len(X_train) + len(X_val) + len(X_test):,}"
)

In [ ]:
# ============================================================
# VERIFY CLASS DISTRIBUTION
# ============================================================

def class_distribution(y_data, name):

    counts = y_data.value_counts().sort_index()
    percentages = (
        counts / len(y_data) * 100
    )

    result = pd.DataFrame({
        "Count": counts,
        "Percentage": percentages.round(2)
    })

    print(f"\n{name}")
    display(result)

    return result


train_dist = class_distribution(
    y_train,
    "TRAIN"
)

val_dist = class_distribution(
    y_val,
    "VALIDATION"
)

test_dist = class_distribution(
    y_test,
    "TEST"
)

In [ ]:
# ============================================================
# SAVE AML DATA SPLITS
# ============================================================

AML_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

X_train.to_parquet(
    AML_DATA_DIR / "aml_X_train.parquet",
    index=False
)

X_val.to_parquet(
    AML_DATA_DIR / "aml_X_val.parquet",
    index=False
)

X_test.to_parquet(
    AML_DATA_DIR / "aml_X_test.parquet",
    index=False
)

y_train.to_frame("grade").to_parquet(
    AML_DATA_DIR / "aml_y_train.parquet",
    index=False
)

y_val.to_frame("grade").to_parquet(
    AML_DATA_DIR / "aml_y_val.parquet",
    index=False
)

y_test.to_frame("grade").to_parquet(
    AML_DATA_DIR / "aml_y_test.parquet",
    index=False
)

print("AML train/validation/test datasets saved.")

In [ ]:
# ============================================================
# SAVE FEATURE DEFINITIONS
# ============================================================

feature_definition = pd.DataFrame({
    "Feature": X.columns,
    "Type": [
        "Original"
        if col in BASE_FEATURES
        else "Engineered"
        for col in X.columns
    ]
})

display(feature_definition)

feature_definition.to_csv(
    PROJECT_ROOT
    / "results"
    / "tables"
    / "aml_feature_definition.csv",
    index=False
)